# Datenvalidierung in Python (1 Stunde)  
**VHS – Kurs: Data Science / Deep-Learning-Vorbereitung (Tag 1)**

Heute nach der Datenbereinigung kommt der nächste Schritt: **Datenvalidierung**.  
Ziel: Bevor wir EDA machen, prüfen wir systematisch, ob die Daten **plausibel, konsistent und analysierbar** sind.


## Agenda (ca. 60 Minuten)

1. **Vorstellung der wichtigsten Validierungsfunktionen** (mit Mini-Beispielen)  
2. **Zusammenhang / Pipeline**: Wie hängt Validierung mit Bereinigung & EDA zusammen?  
3. **Mini‑Projekt**: „Datencheck vor EDA“ (ihr baut einen Validierungs‑Bericht)

> Danach im Kurs: **EDA (Explorative Datenanalyse)** – aber nur, wenn die Daten vertrauenswürdig sind.


## 0) Setup

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

# Reproduzierbarkeit
rng = np.random.default_rng(42)


## Was ist Datenvalidierung?

Datenvalidierung bedeutet: **Regeln (Prüfungen)** definieren und prüfen, ob die Daten diese Regeln erfüllen.

Typische Fragen:
- Gibt es fehlende Werte in Pflichtfeldern?
- Stimmen Datentypen (Zahl vs. Text vs. Datum)?
- Liegen Werte in plausiblen Bereichen (z. B. Alter 0–120)?
- Sind IDs eindeutig?
- Sind Kategorien/Labels gültig?
- Sind Felder untereinander konsistent (z. B. `zahlungsstatus="bezahlt"` ⇒ `betrag > 0`)?

**Warum vor EDA?**
- EDA auf „kaputten“ Daten führt zu falschen Erkenntnisse.
- Validierung macht Fehler **sichtbar** und **messbar** (wie viele, wo, wie schlimm).


## 1) Übungsdaten erzeugen (mit absichtlichen Fehlern)

Wir simulieren Daten aus einem kleinen Kurs‑/Anmeldesystem.


In [ ]:
def make_demo_data(n=30, seed=42):
    rng = np.random.default_rng(seed)
    kurs = rng.choice(["Python-Grundlagen", "Datenvisualisierung", "DL-Einführung"], size=n, replace=True)
    zahlungsstatus = rng.choice(["bezahlt", "offen", "storniert"], size=n, replace=True, p=[0.6, 0.3, 0.1])

    df = pd.DataFrame({
        "teilnehmer_id": [f"T{1000+i}" for i in range(n)],
        "name": rng.choice(["Ali", "Mia", "Noah", "Lina", "Ben", "Sara", "Omar", "Lea"], size=n),
        "email": [f"user{i}@example.com" for i in range(n)],
        "alter": rng.integers(16, 65, size=n),
        "kurs": kurs,
        "kursdatum": pd.to_datetime("2026-03-10") + pd.to_timedelta(rng.integers(-3, 10, size=n), unit="D"),
        "betrag_eur": np.round(rng.normal(79, 15, size=n), 2),
        "zahlungsstatus": zahlungsstatus,
        "stunden": rng.integers(1, 9, size=n)
    })

    # --- absichtliche Fehler einbauen ---
    df.loc[2, "alter"] = -5                 # unplausibel
    df.loc[5, "alter"] = 150                # unplausibel
    df.loc[7, "email"] = "not-an-email"     # falsches Format
    df.loc[9, "kurs"] = "Data Vizz"         # ungültige Kategorie
    df.loc[11, "betrag_eur"] = -20          # unplausibel
    df.loc[12, "kursdatum"] = "2026/99/99"  # kaputtes Datum (String)
    df.loc[13, "teilnehmer_id"] = df.loc[0, "teilnehmer_id"]  # Duplikat
    df.loc[15, "stunden"] = np.nan          # Missing
    df.loc[18, "zahlungsstatus"] = "paid"   # ungültig
    df.loc[20, "zahlungsstatus"] = "bezahlt"
    df.loc[20, "betrag_eur"] = 0            # Inkonsistenz: bezahlt, aber 0
    df.loc[23, "name"] = None               # Missing in Pflichtfeld
    df.loc[25, "betrag_eur"] = "79 EUR"     # falscher Datentyp
    
    return df

df = make_demo_data()
df.head(10)


## 2) Vorstellung: Validierungsfunktionen (mit Mini‑Beispielen)

Wir bauen kleine, **verständliche** Prüfungen.  
Jeder Check liefert entweder:
- eine **Maske** (True/False pro Zeile) oder
- eine **Liste von Fehlerzeilen** oder
- eine **Bericht‑Zeile** (Wie viele Fehler?)

> Didaktik: Erst einzelne Bausteine → danach eine Pipeline.


### 2.1 Pflichtfelder: Fehlende Werte prüfen

In [ ]:
def check_required(df, cols):
    """Prüft, ob in den Spalten cols fehlende Werte sind."""
    missing = df[cols].isna()
    summary = missing.sum().sort_values(ascending=False)
    return summary

check_required(df, cols=["teilnehmer_id", "name", "email", "kurs", "kursdatum", "betrag_eur", "zahlungsstatus"])


### 2.2 Eindeutigkeit: Duplikate prüfen (z. B. IDs)

In [ ]:
def check_unique(df, col):
    dup_mask = df[col].duplicated(keep=False)
    return df.loc[dup_mask, [col]]

check_unique(df, "teilnehmer_id")


### 2.3 Datentypen: numerisch / Datum erzwingen und prüfen

In [ ]:
def konvertieren_numeric(series):
    """Versucht, eine Series in numerisch zu konvertieren; Fehler werden zu NaN."""
    return pd.to_numeric(series, errors="konvertieren")

def konvertieren_datetime(series):
    """Versucht, eine Series in Datum zu konvertieren; Fehler werden zu NaT."""
    return pd.to_datetime(series, errors="konvertieren")

df_tmp = df.copy()
df_tmp["betrag_num"] = konvertieren_numeric(df_tmp["betrag_eur"])
df_tmp["kursdatum_dt"] = konvertieren_datetime(df_tmp["kursdatum"])

df_tmp[["betrag_eur", "betrag_num", "kursdatum", "kursdatum_dt"]].head(15)


### 2.4 Wertebereiche: Plausibilitätsgrenzen prüfen

In [ ]:
def check_range(df, col, min_val=None, max_val=None):
    s = df[col]
    mask = pd.Series(True, index=df.index)
    if min_val is not None:
        mask &= (s >= min_val)
    if max_val is not None:
        mask &= (s <= max_val)
    invalid = df.loc[~mask, [col]]
    return invalid

# Beispiel: Alter 0-120 (erst in numeric umwandeln)
check_range(df.assign(alter_num=konvertieren_numeric(df["alter"])), "alter_num", 0, 120)


### 2.5 Kategorien: Nur erlaubte Werte zulassen

In [ ]:
def check_allowed_values(df, col, allowed):
    mask = df[col].isin(allowed)
    return df.loc[~mask, [col]]

check_allowed_values(df, "zahlungsstatus", allowed={"bezahlt", "offen", "storniert"})


### 2.6 Format-Regeln: z. B. E‑Mail per Regex

In [ ]:
EMAIL_REGEX = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

def check_email_format(df, col="email"):
    s = df[col].astype("string")
    mask = s.apply(lambda x: bool(EMAIL_REGEX.match(x)) if pd.notna(x) else False)
    return df.loc[~mask, [col]]

check_email_format(df)


### 2.7 Spaltenübergreifende Konsistenz: Regeln über mehrere Spalten

In [ ]:
def check_paid_amount_consistency(df, status_col="zahlungsstatus", amount_col="betrag_eur"):
    amount = konvertieren_numeric(df[amount_col])
    status = df[status_col].astype("string")
    
    # Regel: Wenn bezahlt -> Betrag muss > 0 sein
    mask_ok = ~((status == "bezahlt") & (amount <= 0))
    return df.loc[~mask_ok, [status_col, amount_col]]

check_paid_amount_consistency(df)


## 3) Zusammenhang: Wie hängt das zusammen? (Bereinigung → Validierung → EDA)

**Datenbereinigung** (Bereinigung) heißt z. B.:  
- fehlende Werte behandeln, Typen konvertieren, Spalten umbenennen, Ausreißer ggf. markieren, Duplikate entfernen

**Datenvalidierung** heißt:  
- **Regeln festlegen** und **prüfen**, ob Daten diese Regeln erfüllen (und wo nicht)

**Warum ist Validierung ein eigener Schritt?**
- Bereinigung ist oft „Heuristik“ (es gibt mehrere mögliche Lösungen)
- Validierung ist „Qualitätsmessung“ (Regeln + Messzahlen + Bericht)

Praktisch:
- Wir schreiben Prüfungen → erhalten einen Bericht → entscheiden Maßnahmen → erst dann EDA.


## 4) Mini‑Projekt: Baue einen Validierungs‑Bericht (Datencheck vor EDA)

### Aufgabenstellung (20–25 Min)

Ihr bekommt den DataFrame `df` (Anmeldedaten).  
Ziel: Erstellt einen **Validierungs‑Bericht** als Tabelle mit Spalten:

- `pruefung_name`
- `anzahl_verstoesse` (wie viele Zeilen verletzen die Regel?)
- `beispiel_zeilen` (z. B. die ersten 3 Zeilenindizes als Liste)

Validierungsregeln (mindestens diese 6):
1) Pflichtfelder nicht leer: `teilnehmer_id, name, email, kurs, kursdatum, betrag_eur, zahlungsstatus`  
2) `teilnehmer_id` ist eindeutig  
3) `alter` in [0, 120]  
4) `stunden` in [1, 8]  
5) `zahlungsstatus` ∈ {bezahlt, offen, storniert}  
6) E-Mail Format gültig  
7) Bonus: Wenn `zahlungsstatus == "bezahlt"` ⇒ `betrag_eur > 0`

> Tipp: Für Zahlen/Daten zuerst **konvertieren** (to_numeric / to_datetime), damit Prüfungen stabil laufen.


### 4.1 Gerüst: Hilfsfunktion für Bericht-Zeilen

In [ ]:
def make_report_row(pruefung_name, bad_index, max_examples=3):
    bad_index = list(bad_index)
    return {
        "pruefung_name": check_name,
        "anzahl_verstoesse": len(bad_index),
        "beispiel_zeilen": bad_index[:max_examples]
    }


### 4.2 Lösung (Pipeline) – mit Kommentaren

In [ ]:
report_rows = []

# 1) Pflichtfelder
required_cols = ["teilnehmer_id", "name", "email", "kurs", "kursdatum", "betrag_eur", "zahlungsstatus"]
missing_any = df[required_cols].isna().any(axis=1)
report_rows.append(make_report_row("Pflichtfelder: fehlende Werte", df.index[missing_any]))

# 2) eindeutige ID
dup_id = df["teilnehmer_id"].duplicated(keep=False)
report_rows.append(make_report_row("ID eindeutig: teilnehmer_id", df.index[dup_id]))

# 3) Alter Bereich
alter_num = konvertieren_numeric(df["alter"])
bad_alter = (alter_num < 0) | (alter_num > 120) | (alter_num.isna())
report_rows.append(make_report_row("Plausibilität: alter in [0,120]", df.index[bad_alter]))

# 4) Stunden Bereich
stunden_num = konvertieren_numeric(df["stunden"])
bad_stunden = (stunden_num < 1) | (stunden_num > 8) | (stunden_num.isna())
report_rows.append(make_report_row("Plausibilität: stunden in [1,8]", df.index[bad_stunden]))

# 5) Zahlungsstatus erlaubt
allowed_status = {"bezahlt", "offen", "storniert"}
bad_status = ~df["zahlungsstatus"].isin(allowed_status)
report_rows.append(make_report_row("Kategorien: zahlungsstatus gültig", df.index[bad_status]))

# 6) E-Mail Format
email_str = df["email"].astype("string")
bad_email = ~email_str.apply(lambda x: bool(EMAIL_REGEX.match(x)) if pd.notna(x) else False)
report_rows.append(make_report_row("Format: email regex", df.index[bad_email]))

# 7) Bonus: bezahlt -> Betrag > 0 (nach numeric coercion)
betrag_num = konvertieren_numeric(df["betrag_eur"])
paid = (df["zahlungsstatus"] == "bezahlt")
bad_paid_amount = paid & (betrag_num <= 0)
report_rows.append(make_report_row("Konsistenz: bezahlt => betrag > 0", df.index[bad_paid_amount]))

report = pd.DataFrame(report_rows).sort_values("anzahl_verstoesse", ascending=False).reset_index(drop=True)
report


### 4.3 Detailansicht: Zeige fehlerhafte Zeilen pro Check

In [ ]:
def show_bad_rows(df, bad_index, cols=None, max_rows=10):
    out = df.loc[bad_index]
    if cols is not None:
        out = out[cols]
    return out.head(max_rows)

# Beispiel: Welche Zeilen haben ungültige Emails?
bad_email_idx = df.index[~df["email"].astype("string").apply(lambda x: bool(EMAIL_REGEX.match(x)) if pd.notna(x) else False)]
show_bad_rows(df, bad_email_idx, cols=["teilnehmer_id", "name", "email"])


## 5) (Optional) Mini-Ausblick: Was machen wir mit den Fehlern?

In echten Projekten gibt es typischerweise drei Strategien:

1) **Korrigieren** (z. B. Typen sauber konvertieren, Schreibfehler in Kategorien mappen)  
2) **Entfernen** (z. B. klare Duplikate oder kaputte Zeilen, wenn erlaubt)  
3) **Markieren & später behandeln** (z. B. Ausreißer flaggen, „unknown“ Kategorien)

Wichtig: Validierung heißt nicht automatisch „alles löschen“.  
Es heißt: **Transparenz schaffen**, bevor wir interpretieren.

➡️ Nächster Schritt im Kurs: **EDA** (Histogramme, Boxplots, Wertehäufigkeiten, erste Erkenntnisse).
